# Dark Current vs Proton Fluence in 4H-SiC Detector

This notebook predicts how dark current evolves under proton irradiation in a 10 um
4H-SiC epitaxial detector using an additive delta-J model:

$$J_{\mathrm{dark}}(\Phi) = J_{\mathrm{dark}}(0) + \Delta J(\Phi)$$

where $J_{\mathrm{dark}}(0)$ is the v1.1 calibrated pristine baseline (~111 pA at $V = -30$ V,
$A = 0.04$ cm$^2$) and $\Delta J(\Phi)$ arises from radiation-degraded lifetimes driving increased
SRH bulk generation, TAT (trap-assisted tunneling) effective generation, and surface recombination.

**Counterintuitive SiC behavior:** Unlike silicon, where dark current always increases monotonically
with fluence, 4H-SiC dark current may plateau or even decrease at very high fluence. This occurs
because carrier removal reduces the effective doping, shrinking the depletion width and altering
the generation volume geometry. The interplay between increasing defect concentrations and
decreasing generation volume produces a non-trivial fluence dependence unique to wide-bandgap
semiconductors.

**Sections:**
1. Dark current vs fluence sweep with component decomposition
2. Total dark current plot (using `plot_dark_current_vs_fluence`)
3. Delta-J analysis (radiation-induced increase above baseline)
4. Multi-bias comparison ($V = -10, -30, -50$ V)
5. Discussion
6. Summary table

**Reference:** Burin et al., arXiv:2407.16710 (2024) -- damage constants for 4H-SiC

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.dark_current import dark_current_vs_fluence, plot_dark_current_vs_fluence
from src.radiation_damage import RadiationDamageParams

# Publication-quality settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'figure.figsize': (8, 6),
})

print('Notebook 05: Dark Current vs Proton Fluence')
print('Device: 10 um 4H-SiC epitaxial detector')
print('Beam: 62 MeV protons')
print('Area: 0.04 cm^2 (4 mm^2 dosimetry detector)')

In [ ]:
# Dark current vs fluence sweep
# Include fluence=0 as first point for baseline/delta-J decomposition
fluence_range = np.concatenate([[0.0], np.geomspace(1e10, 5e13, 12)])

print(f'Fluence grid: {len(fluence_range)} points (pristine + {len(fluence_range)-1} irradiated)')
for i, phi in enumerate(fluence_range):
    print(f'  [{i:2d}] {phi:.2e} cm^-2')

# Run sweep at standard operating point
result = dark_current_vs_fluence(
    fluence_range,
    V_bias=-30.0,
    area=0.04,
    lifetime_model='linear',
)

# Summary table
print()
print('=' * 60)
print(f'{"Fluence (p/cm2)":>18s}  {"I_total (pA)":>14s}  {"delta_I (pA)":>14s}')
print('-' * 60)
for i in range(len(result['fluences'])):
    phi = result['fluences'][i]
    I_tot = abs(result['I_total'][i]) * 1e12  # A -> pA
    dI = abs(result['delta_I'][i]) * 1e12 if 'delta_I' in result else float('nan')
    print(f'{phi:>18.2e}  {I_tot:>14.3f}  {dI:>14.3f}')
print('=' * 60)
print(f'Baseline (pristine): {abs(result["I_baseline"]) * 1e12:.3f} pA')

In [ ]:
# Figure 1: Total dark current vs fluence with component decomposition
import os
os.makedirs('../figures', exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 6))
plot_dark_current_vs_fluence(result, ax=ax)
fig.tight_layout()
fig.savefig('../figures/dark_current_vs_fluence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/dark_current_vs_fluence.png')

In [ ]:
# Figure 2: Delta-J analysis -- radiation-induced dark current increase
# Shows only the increase above pristine baseline
fig, ax = plt.subplots(figsize=(8, 6))

fluences = np.asarray(result['fluences'])
delta_I = np.asarray(result['delta_I'])

# Skip fluence=0 for log-log (delta_I=0 at pristine)
mask = fluences > 0
abs_delta = np.abs(delta_I[mask])

# Only plot points where delta_I > 0
valid = abs_delta > 0
if np.any(valid):
    ax.loglog(fluences[mask][valid], abs_delta[valid], 'o-',
             color='#d62728', markersize=6, linewidth=2,
             label=r'$|\Delta J|$ (radiation-induced)')

# Annotate baseline
I_bl_pA = abs(result['I_baseline']) * 1e12
ax.axhline(abs(result['I_baseline']), color='gray', linestyle='--',
           linewidth=1.0, label=f'Pristine baseline ({I_bl_pA:.1f} pA)')

ax.set_xlabel(r'Proton Fluence (p/cm$^2$)')
ax.set_ylabel(r'$|\Delta I_{\mathrm{dark}}|$ (A)')
ax.set_title('Radiation-Induced Dark Current Increase (Delta-J)')
ax.grid(True, alpha=0.3)
ax.legend(loc='best')
fig.tight_layout()
fig.savefig('../figures/dark_current_delta_j.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/dark_current_delta_j.png')

In [ ]:
# Figure 3: Multi-bias comparison
# Shows how dark current depends on both fluence and operating voltage
bias_voltages = [-10.0, -30.0, -50.0]
colors = ['#d62728', '#1f77b4', '#2ca02c']

# Use a subset of fluences for faster computation
fluence_multibias = np.concatenate([[0.0], np.geomspace(1e10, 5e13, 10)])

results_multibias = {}
for V in bias_voltages:
    print(f'Computing dark current vs fluence at V = {V} V ...')
    results_multibias[V] = dark_current_vs_fluence(
        fluence_multibias,
        V_bias=V,
        area=0.04,
        lifetime_model='linear',
    )

fig, ax = plt.subplots(figsize=(8, 6))
for V, color in zip(bias_voltages, colors):
    r = results_multibias[V]
    fl = np.asarray(r['fluences'])
    I_tot = np.abs(np.asarray(r['I_total']))
    # Skip fluence=0 for log scale
    mask = fl > 0
    ax.loglog(fl[mask], I_tot[mask], 'o-',
             color=color, markersize=5, linewidth=1.8,
             label=f'$V = {V:.0f}$ V')

ax.set_xlabel(r'Proton Fluence (p/cm$^2$)')
ax.set_ylabel('|Dark Current| (A)')
ax.set_title('Dark Current vs Fluence at Multiple Bias Voltages')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('../figures/dark_current_multibias.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/dark_current_multibias.png')

## Discussion

### Key Findings

**Dark current evolution with fluence:**
- At low fluences ($< 10^{11}$ p/cm$^2$), dark current changes minimally from the pristine
  baseline, consistent with negligible lifetime degradation at these damage levels.
- At moderate fluences ($10^{12}$--$10^{13}$ p/cm$^2$), radiation-induced dark current
  ($\Delta J$) becomes a measurable fraction of the baseline, driven primarily by increased
  effective generation via the TAT mechanism.
- The total dark current signal is dominated by the effective $N_t$-based generation rate
  rather than intrinsic carrier concentration ($n_i^2$), which is extremely small in 4H-SiC
  ($n_i \approx 10^{-9}$ cm$^{-3}$ at 300 K).

### SiC-Specific Physics vs Silicon

**4H-SiC:** Dark current is dominated by TAT effective generation through mid-gap defects.
The ultra-small intrinsic carrier concentration means that thermal generation ($\propto n_i$)
is negligible. At very high fluence, carrier removal reduces effective doping, which changes
the depletion geometry and field profile. This can cause dark current to plateau or even
decrease -- a counterintuitive result that arises because the generation volume shrinks
faster than defect concentrations increase.

**Silicon:** Dark current always increases monotonically with fluence because $n_i$ is large
enough that thermal generation dominates, and the generation volume expands with damage
(type inversion extends depletion).

### Bias Voltage Dependence

Higher reverse bias increases dark current at all fluence levels because:
1. The depletion width extends further into the bulk, increasing the generation volume
2. Higher electric fields enhance TAT generation rates
3. Surface recombination velocity contributions scale with field at contacts

### Preserved Calibration

The additive delta-J model preserves the v1.1 calibrated pristine baseline by construction:
at $\Phi = 0$, the simulation uses the calibrated device parameters and returns exactly
$J_{\mathrm{dark}}(0)$ without any radiation-induced corrections. This ensures backward
compatibility with all pristine dark current predictions.

### Limitations

- NIEL hardness factor for 62 MeV protons ($\kappa = 0.35$) is interpolated from
  available data; validation against SR-NIEL calculator is pending
- Uniform doping profile approximation in the bulk; graded epi profile effects on
  radiation-induced dark current generation are a future study
- Carrier removal rate at 62 MeV is interpolated (no direct measurement)

In [ ]:
# Summary table: component decomposition at all fluence points
print('=' * 90)
print('Dark Current Component Decomposition (V = -30 V, area = 0.04 cm^2, 62 MeV protons)')
print('=' * 90)
header = (f'{"Fluence (p/cm2)":>18s}  {"I_total (pA)":>12s}  {"I_SRH (pA)":>12s}  '
          f'{"I_TAT (pA)":>12s}  {"I_SRV (pA)":>12s}  {"delta_I (pA)":>12s}')
print(header)
print('-' * 90)

for i in range(len(result['fluences'])):
    phi = result['fluences'][i]
    I_tot = abs(result['I_total'][i]) * 1e12
    I_srh = abs(result['I_SRH'][i]) * 1e12
    I_tat = abs(result['I_TAT'][i]) * 1e12
    I_srv = abs(result['I_SRV'][i]) * 1e12
    dI = abs(result['delta_I'][i]) * 1e12 if 'delta_I' in result else float('nan')
    print(f'{phi:>18.2e}  {I_tot:>12.3f}  {I_srh:>12.3f}  '
          f'{I_tat:>12.3f}  {I_srv:>12.3f}  {dI:>12.3f}')

print('=' * 90)
print(f'Baseline (pristine): {abs(result["I_baseline"]) * 1e12:.3f} pA')
print(f'Lifetime model: {result["lifetime_model"]}')
print(f'Bias voltage: {result["V_bias"]} V')
print(f'Energy: {result["energy_MeV"]} MeV protons')